# HG4052 · Week 7 Practical
## Slopes, descent, and a vowel classifier

**No installs: numpy, pandas, matplotlib and scikit-learn are preinstalled on Colab. No audio this week: the data are formant measurements.**

By the end you will have:
- ✅ a slope measured by nudging, checked against the calculus recipe
- ✅ gradient descent with the update line written by you, run at three step sizes
- ✅ 1,668 real vowels on the vowel chart: 12 vowels, 139 speakers (Hillenbrand et al. 1995)
- ✅ one neuron separating /i/ from /ɑ/, with its two weights read phonetically
- ✅ a network trained on all 12 vowels, a learning rate that breaks it, and a confusion matrix on speakers it never saw

**How this notebook works.** Same as every week: click a cell, press **Shift + Enter**, and read the output underneath. The one **✏️ TODO** in code is a single line. Written TODOs are answered by double-clicking the cell and typing.

**Short on time?** Parts 1, 4 and 5 are the core. Part 2 is one plot; Part 3 is five minutes; Part 6 is take-home.

---
## 0 · Setup

In [ ]:
!pwd

In [ ]:
import os, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import ConfusionMatrixDisplay

warnings.filterwarnings("ignore")        # scikit-learn warns when training stops at max_iter; that is expected here
np.set_printoptions(precision=3, suppress=True)

os.makedirs("data", exist_ok=True)
CSV = "data/hillenbrand1995_vowels.csv"
if not os.path.exists(CSV):
    try:
        urllib.request.urlretrieve("https://raw.githubusercontent.com/chenchenzi/hg4052-materials/main/week07/hillenbrand1995_vowels.csv", CSV)
    except Exception as e:
        print("download failed:", e)
ok = os.path.exists(CSV) and os.path.getsize(CSV) > 1000
print(("✅" if ok else "❌"), "hillenbrand1995_vowels.csv", (f"{os.path.getsize(CSV):,} bytes" if ok else "missing (also on NTULearn: download it and upload into data/)"))

---
## 1 · Slopes and steps

**Nudging.** The slope of f(x) = x² at a point: move x by a tiny amount h and see how far f moves. The calculus recipe says the slope is 2x; compare.

In [ ]:
def f(x):
    return x ** 2                      # ** means "to the power of"

h = 0.001
for x in [1, 2, 3]:
    nudged = (f(x + h) - f(x)) / h
    print(f"x = {x}:  slope by nudging = {nudged:.3f}   recipe 2x = {2 * x}")

**Gradient descent.** Start at x = 3 and repeat one move: step against the slope. The step size is η (eta), the learning rate.

**✏️ TODO (one line).** Write the update line from the lecture inside `descend`: the new x is the old x minus eta times the slope. If you leave it, a finished copy takes over so the rest of the notebook runs.

In [ ]:
def descend(eta, x=3.0, steps=10):
    xs = [x]
    for _ in range(steps):
        slope = 2 * x                  # the recipe for the slope of x²
        # ✏️ TODO: the update line.   Building blocks:  x   eta   slope
        x = ...
        xs.append(x)
    return xs

def descend_reference(eta, x=3.0, steps=10):
    xs = [x]
    for _ in range(steps):
        x = x - eta * (2 * x)
        xs.append(x)
    return xs

try:
    TODO_DONE = abs(descend(0.1)[1] - 2.4) < 1e-9
except TypeError:
    TODO_DONE = False
DESCEND = descend if TODO_DONE else descend_reference
print("✅ your update line works" if TODO_DONE else "⬆ TODO not finished: using descend_reference for now")

runs = {eta: DESCEND(eta) for eta in (0.1, 0.45, 1.1)}
for eta, xs in runs.items():
    print(f"eta = {eta:<4}  x: " + "  ".join(f"{v:6.2f}" for v in xs[:7]) + "  ...  " + f"{xs[-1]:.2f}")

grid = np.linspace(-6, 6, 200)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, (eta, xs) in zip(axes, runs.items()):
    ax.plot(grid, grid ** 2, color="0.7")
    shown = [v for v in xs if abs(v) <= 6]
    ax.plot(shown, [v ** 2 for v in shown], "o-", color="#d9472b")
    ax.set_title(f"eta = {eta}"); ax.set_xlabel("x"); ax.set_ylim(-1, 38)
axes[0].set_ylabel("f(x) = x²")
plt.tight_layout(); plt.show()

**✏️ TODO (in words, double-click).** Describe each run in one phrase (crawls, settles, or diverges?). For eta = 1.1, what happens to the distance from the minimum at every step, and why?

> **...**

---
## 2 · The vowel chart

Hillenbrand et al. (1995) recorded 139 speakers from Michigan (45 men, 48 women, 46 children) saying 12 vowels in /hVd/ words (*heed, hid, hayed, head, had, hod, hawed, hoed, hood, who'd, hud, heard*), and measured each vowel's formants. One row per token.

In [ ]:
df = pd.read_csv(CSV)
print(df.shape, "·", df["speaker"].nunique(), "speakers")
print(df["group"].value_counts().to_dict())
df.head()

In [ ]:
VOWELS = ["i", "ɪ", "e", "ɛ", "æ", "ɑ", "ɔ", "o", "ʊ", "u", "ʌ", "ɝ"]
fig, ax = plt.subplots(figsize=(7.5, 6))
for v in VOWELS:
    sub = df[df["vowel"] == v]
    ax.scatter(sub["f2"], sub["f1"], s=8, alpha=0.35)
    ax.text(sub["f2"].mean(), sub["f1"].mean(), v, fontsize=18, ha="center", va="center", weight="bold")
ax.invert_xaxis(); ax.invert_yaxis()                 # the phonetician's orientation: front vowels left, high vowels top
ax.set_xlabel("F2 (Hz)"); ax.set_ylabel("F1 (Hz)"); ax.set_title("12 vowels, 139 speakers")
plt.show()

---
## 3 · One neuron: /i/ vs /ɑ/

A logistic regression is one neuron: a weighted sum of the inputs plus a bias, then the sigmoid. The inputs are F1 and F2, rescaled to a common scale (mean 0, spread 1) so that the two weights can be compared.

In [ ]:
pair = df[df["vowel"].isin(["i", "ɑ"])]
X2 = pair[["f1", "f2"]].values
y2 = (pair["vowel"] == "ɑ").astype(int).values       # 1 = /ɑ/, 0 = /i/
scale2 = StandardScaler().fit(X2)
neuron = LogisticRegression().fit(scale2.transform(X2), y2)
w_f1, w_f2 = neuron.coef_[0]
print(f"weight on F1: {w_f1:+.2f}   weight on F2: {w_f2:+.2f}   bias: {neuron.intercept_[0]:+.2f}")
print(f"accuracy on these {len(pair)} tokens: {neuron.score(scale2.transform(X2), y2):.1%}")

f1g, f2g = np.meshgrid(np.linspace(200, 1400, 200), np.linspace(600, 3600, 200))
p = neuron.predict_proba(scale2.transform(np.c_[f1g.ravel(), f2g.ravel()]))[:, 1].reshape(f1g.shape)
fig, ax = plt.subplots(figsize=(6.5, 5))
ax.contour(f2g, f1g, p, levels=[0.5], colors="k")
for v, c in (("i", "#0e6f66"), ("ɑ", "#d9472b")):
    sub = pair[pair["vowel"] == v]
    ax.scatter(sub["f2"], sub["f1"], s=10, color=c, label=v)
ax.invert_xaxis(); ax.invert_yaxis(); ax.legend()
ax.set_xlabel("F2 (Hz)"); ax.set_ylabel("F1 (Hz)"); ax.set_title("the neuron's boundary (black line): P(/ɑ/) = 0.5")
plt.show()

**✏️ TODO (in words, double-click).** Read the two weights phonetically: which sign goes with F1 and which with F2, and what does each say about how /ɑ/ differs from /i/ in the mouth?

> **...**

---
## 4 · A network on all 12 vowels

Now all 12 vowels, with one hidden layer of 16 units. The test at the end must be on speakers the network never saw, so we set 35 speakers aside **before** training.

In [ ]:
speakers = np.array(sorted(df["speaker"].unique()))
test_speakers = np.random.default_rng(7).choice(speakers, size=35, replace=False)
held_out = df["speaker"].isin(test_speakers).values
X = df[["f1", "f2"]].values
y = df["vowel"].values
scale = StandardScaler().fit(X[~held_out])            # the scale is learned from the training speakers only
X_train, y_train = scale.transform(X[~held_out]), y[~held_out]
X_test,  y_test  = scale.transform(X[held_out]),  y[held_out]
print(f"training: {len(y_train)} tokens from {139 - 35} speakers · held out: {len(y_test)} tokens from 35 speakers")

def train(eta):
    net = MLPClassifier(hidden_layer_sizes=(16,), solver="sgd", learning_rate_init=eta,
                        max_iter=200, n_iter_no_change=200, random_state=0)
    return net.fit(X_train, y_train)

net = train(0.05)
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(net.loss_curve_, label="eta = 0.05")
ax.set_xlabel("epoch (one pass through the training tokens)"); ax.set_ylabel("loss (cross-entropy)"); ax.legend()
plt.show()
print(f"final training loss {net.loss_curve_[-1]:.2f}")

In [ ]:
wild = train(5.0)                                      # a learning rate 100 times larger
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(net.loss_curve_, label="eta = 0.05")
ax.plot(wild.loss_curve_, label="eta = 5.0 (100 times larger)")
ax.axhline(np.log(12), color="0.5", ls="--", label="ln 12: guessing uniformly among 12 vowels")
ax.set_xlabel("epoch"); ax.set_ylabel("loss (cross-entropy)"); ax.legend()
plt.show()
print(f"eta = 5.0: final loss {wild.loss_curve_[-1]:.2f}, training accuracy {wild.score(X_train, y_train):.0%}")

**✏️ TODO (in words, double-click).** Describe the two loss curves. Why does the eta = 5.0 run end near the dashed line, and what does that line mean about what the network has learned?

> **...**

---
## 5 · Held-out speakers

In [ ]:
print(f"accuracy on the training speakers: {net.score(X_train, y_train):.1%}")
print(f"accuracy on 35 held-out speakers:  {net.score(X_test, y_test):.1%}   (chance = {1/12:.1%})")
fig, ax = plt.subplots(figsize=(7.5, 6.5))
ConfusionMatrixDisplay.from_predictions(y_test, net.predict(X_test), labels=VOWELS, ax=ax, colorbar=False)
ax.set_title("held-out speakers: rows = the vowel said, columns = the network's answer")
plt.show()

**✏️ TODO (in words, double-click).** Name the two pairs the network confuses most. Find them on the Part 2 chart: why is F1 and F2 alone not enough to tell them apart? What else distinguishes them when you say them?

> **...**

In [ ]:
# Optional, one cell: give the network more cues than F1 and F2.
X5 = df[["f1", "f2", "f3", "f0", "duration_ms"]].values
scale5 = StandardScaler().fit(X5[~held_out])
net5 = MLPClassifier(hidden_layer_sizes=(16,), solver="sgd", learning_rate_init=0.05,
                     max_iter=200, n_iter_no_change=200, random_state=0).fit(scale5.transform(X5[~held_out]), y_train)
print(f"held-out accuracy with F1, F2, F3, F0 and duration: {net5.score(scale5.transform(X5[held_out]), y_test):.1%}")

---
## 6 · Stretch (pick one, take-home)

**(a) Overfit on purpose.** Train a large network on only 10 tokens per vowel, then compare its accuracy on those tokens with its accuracy on the held-out speakers.

In [ ]:
rng = np.random.default_rng(1)
small = np.concatenate([rng.choice(np.where(y_train == v)[0], 10, replace=False) for v in VOWELS])
big = MLPClassifier(hidden_layer_sizes=(200, 200), max_iter=2000, random_state=0).fit(X_train[small], y_train[small])
print(f"on its 120 training tokens: {big.score(X_train[small], y_train[small]):.0%}   on held-out speakers: {big.score(X_test, y_test):.0%}")

**(b) Sweep the hidden-layer size** from 1 to 100 units. Where does adding units stop helping, and what does that say about F1 and F2 as inputs?

In [ ]:
for size in [1, 2, 4, 8, 16, 32, 64, 100]:
    m = MLPClassifier(hidden_layer_sizes=(size,), solver="sgd", learning_rate_init=0.05,
                      max_iter=200, n_iter_no_change=200, random_state=0).fit(X_train, y_train)
    print(f"{size:3} units: training {m.score(X_train, y_train):.0%}   held-out {m.score(X_test, y_test):.0%}")

**(c) Your own vowels.** Measure F1 and F2 of your own /i/, /ɑ/ and /u/ in Praat (Week 3 skills: the middle of each vowel), type them in below, and see what the network answers. Does a model trained on 1990s Michigan English classify your accent?

In [ ]:
mine = {"i": (300, 2300), "ɑ": (750, 1200), "u": (320, 900)}     # ✏️ replace with your own measurements: (F1, F2) in Hz
for said, (f1v, f2v) in mine.items():
    print(f"you said /{said}/  →  the network hears /{net.predict(scale.transform([[f1v, f2v]]))[0]}/")

---
## ✅ Done looks like

- three descent runs described in a phrase each, and the update line written by you
- the vowel chart of 139 speakers, and one neuron's boundary between /i/ and /ɑ/ with its weights read phonetically
- two loss curves, one of which never learns, and why it stops at ln 12
- a confusion matrix on speakers the network never saw, and a phonetic reason for its two biggest confusions

**What you built.** A classifier that learns its categories from data, and a check on speakers it never met. Next week the network is large, the input is the whole log-mel spectrogram, and the output is text.